In [3]:
def spk_matrix_to_ranks_projection(spk_matrix, trigger_neuron_idx): #output is the neuron idx in the place related to the trigger neuron
    """
    Projects spike matrix data into rank-based representations relative to a trigger neuron.

    Parameters:
    -----------
    spk_matrix : ndarray, shape (n_neurons, n_time_points)
        Binary matrix indicating spike occurrences for each neuron over time.

    trigger_neuron_idx : int
        Index of the neuron used as the reference for ranking.

    Returns:
    --------
    ranks : ndarray, shape (n_neurons, n_neurons*2+1)
        Matrix representing the rank of each neuron's spike timing relative to the trigger neuron.
    """
    n_neurons = spk_matrix.shape[0]
    ranks = np.full((n_neurons, n_neurons*2+1), np.nan)  # Shape: [trials x neurons*2+1]
    
    avg_spk_time = []
    for neuron_idx in range(n_neurons):
        spike_times = np.where(spk_matrix[neuron_idx] == 1)[0]
        if len(spike_times) > 0:  
            avg_spk_time.append(np.mean(spike_times))  
        else:
            avg_spk_time.append(np.nan)  
    avg_spk_time = np.array(avg_spk_time) 
    neuron_sorted = np.argsort(avg_spk_time)
    valid_neurons = [neuron_idx for neuron_idx in neuron_sorted if not np.isnan(avg_spk_time[neuron_idx])]


    
    if trigger_neuron_idx in valid_neurons:
        base_rank = valid_neurons.index(trigger_neuron_idx) # base rank gets index of trigger neuron in valid neurons
        ranks[trigger_neuron_idx, base_rank+n_neurons] = 1  # Trigger neuron rank is 0
    else:
        base_rank = None
          

    for relative_rank, neuron_idx in enumerate(valid_neurons): #relative rank is index in valid neurons 
        if base_rank is not None:  
            idx = relative_rank - base_rank + n_neurons
            ranks[neuron_idx, idx] = 1
        else:  
            ranks[neuron_idx, idx] = np.nan # hier dus nog goed naar kijken!!!
    return ranks

In [2]:
def get_weighted_values(test_matrix, pattern_template, window_size=10, step_size=5, first_half = True):
    """
    Computes weighted values for spiking activity in specified time windows.

    Parameters:
    -----------
    test_matrix : array-like, shape (n_samples, 5)
        Each row represents a test run with start and end times.

    pattern_template : ndarray
        Template used for computing match scores with spike ranks.

    window_size : int, optional (default=10)
        Size of the time window for spike analysis.

    step_size : int, optional (default=5)
        Step size for sliding the time window.

    first_half : bool, optional (default=True)
        If True, processes the first half of the test run, otherwise the second half.

    Returns:
    --------
    all_values : ndarray, shape (n_samples, max_time_steps)
        Computed match scores for each time window, padded with NaNs.

    all_time_points : ndarray, shape (n_samples, max_time_steps)
        Corresponding time points for each computed score, padded with NaNs.
    """
    # Initialize lists to store the computed values and time points as rows for each test run
    all_values = []
    all_time_points = []

    for i, val in enumerate(test_matrix):
        if first_half == True:
            start_time = int(val[0])  # Start time for first half
            end_time = int(val[2])  # End time 
        else:
            start_time = int(val[2])  # Start time for second half
            end_time = int(val[4]) 
            
        # Initialize lists to store the values and time points for this test run
        values = []
        time_points = []
        empty_spk_matrix_count = 0
        total_time_points = 0  # To count the number of time steps we process
        
        # Iterate over the time steps using the specified window size and step size
        for t in range(start_time, end_time - window_size + 1, step_size):
            total_time_points += 1  # Increment the total time steps processed
            
           
            spk_matrix = spk_matrix_init(session, t, t + window_size, neuron_ids, session_neurons) # Create the spike matrix for the current window
            spiking_neurons = np.where(spk_matrix.sum(axis=1) > 0)[0]# Determine the trigger neuron (the first neuron that spikes in the window)

            if len(spiking_neurons) == 0: # Skip if no neurons spike in this window
                empty_spk_matrix_count += 1
                values.append(0)  # Store NaN if no spike
                time_points.append(t)  # Store the corresponding time point
                continue
            
            trigger_neuron_idx = spiking_neurons[0]  # Choose the first spiking neuron
            ranks = spk_matrix_to_ranks_projection(spk_matrix, trigger_neuron_idx) # Convert the spike matrix to rank    
           
            if ranks.shape != pattern_template.shape:  # Ensure the ranks and occ_template have the same size
                raise ValueError("Ranks and pattern_template must have the same shape.")

            match_score = np.nansum(ranks * pattern_template)   # Multiply ranks by the pattern template and sum the result
            values.append(match_score)  # Store the match score and time point for this window
            time_points.append(t)
        
        # After processing the test run, calculate the fraction of empty matrices
        if total_time_points > 0:
            fraction_empty = empty_spk_matrix_count / total_time_points
            print(f"Test run {i + 1}: {empty_spk_matrix_count} empty spike matrices out of {total_time_points} total time points. Fraction of empty: {fraction_empty:.4f}")
        else:
            print(f"Test run {i + 1}: No time points were processed. Skipping fraction calculation.")
        # fraction_empty = empty_spk_matrix_count / total_time_points
        # print(f"Test run {i + 1}: {empty_spk_matrix_count} empty spike matrices out of {total_time_points} total time points. Fraction of empty: {fraction_empty:.4f}")
        
        # Append the results for this test run
        all_values.append(values)
        all_time_points.append(time_points)
    
    # Pad the lists so that they all have the same length
    max_length = max(len(values) for values in all_values)  # Find the longest test run's length
    all_values_padded = [values + [np.nan] * (max_length - len(values)) for values in all_values]   # Pad with NaN to make all rows the same length
    all_time_points_padded = [time_points + [np.nan] * (max_length - len(time_points)) for time_points in all_time_points]
    
    all_values = np.array(all_values_padded)
    all_time_points = np.array(all_time_points_padded)
    
    return all_values, all_time_points

In [3]:
def plot_values(values_first_half, time_points_first_half, values_second_half, time_points_second_half):
    for i in range(values_first_half.shape[0]):
        plt.figure(figsize=(4, 2))
        plt.plot(time_points_first_half[i], values_first_half[i], marker='o', linestyle='-', color='b', label=f'Test Run {i + 1} - First Half')
        plt.plot(time_points_second_half[i], values_second_half[i], marker='o', linestyle='-', color='r', label=f'Test Run {i + 1} - First Half')
        # Labeling and styling the plot
        plt.xlabel('Time')
        plt.ylabel('Values')
        plt.title(f'Computed Values vs. Time for Test Run {i + 1}')
        plt.grid(True)
        plt.legend()
        plt.show()

In [5]:
def plot_means_first_second_half(all_values_first_half, all_values_second_half):
    """
    Plots the means for the first and second halves of each test run on the x-axis,
    with the mean values on the y-axis, using two points for each test run: one for the first half
    and one for the second half.
    
    Parameters:
    - all_values_first_half: 2D array of values for the first half of each test run.
    - all_values_second_half: 2D array of values for the second half of each test run.
    """
    # Calculate the mean for each test run in the first and second halves
    means_first_half = np.nanmean(all_values_first_half, axis=1)
    means_second_half = np.nanmean(all_values_second_half, axis=1)
    
    plt.figure(figsize=(6, 4))
    x_positions = [0, 1]  # Use 0 for 'First Half' and 1 for 'Second Half'
    for i in range(len(means_first_half)):
        plt.scatter(x_positions[0], means_first_half[i], color='blue', label='First Half' if i == 0 else "")  # Plot first half value
        plt.scatter(x_positions[1], means_second_half[i], color='red', label='Second Half' if i == 0 else "")  # Plot second half value

    for i in range(len(means_first_half)):  # Connect the points for the same test run with a line
        plt.plot([x_positions[0], x_positions[1]], [means_first_half[i], means_second_half[i]], color='black', linewidth=1)
        
    plt.xticks(x_positions, ['First Half', 'Second Half'])  # Set the x-tick labels to 'First Half' and 'Second Half'
    plt.xlabel('Track Half')
    plt.ylabel('Mean Value')
    plt.title('Mean Values for First and Second Halves of Test Runs')
    plt.legend()
    plt.grid(True, axis='y')
    plt.tight_layout()
    plt.show()

In [6]:
def calculate_sem(values):
    """
    Calculate the standard error of the mean (SEM) for the given values.
    
    Parameters:
    - values: A list or array of values for which to calculate the SEM.
    
    Returns:
    - SEM value.
    """
    # Avoid empty slices or NaN values
    non_nan_values = values[~np.isnan(values)]  # Remove NaNs
    if len(non_nan_values) > 1:  # Ensure there are enough values to calculate SEM
        return np.nanstd(non_nan_values) / np.sqrt(len(non_nan_values))
    else:
        return np.nan  # Return NaN if not enough values to compute SEM

def plot_means_sem_first_second_half(all_values_first_half, all_values_second_half):
    """
    Plots the means and standard error of the mean (SEM) for the first and second halves
    of each test run, using box plots and points for each test run.
    
    Parameters:
    - all_values_first_half: 2D array of values for the first half of each test run.
    - all_values_second_half: 2D array of values for the second half of each test run.
    """
    # Calculate the means for each test run in the first and second halves
    means_first_half = np.nanmean(all_values_first_half, axis=1)
    means_second_half = np.nanmean(all_values_second_half, axis=1)
    
    # Calculate the SEM for each test run in the first and second halves
    sem_first_half = np.array([calculate_sem(values) for values in all_values_first_half])
    sem_second_half = np.array([calculate_sem(values) for values in all_values_second_half])

    # Clean the data: Replace NaN with 0 (or handle differently based on your needs)
    sem_first_half_clean = np.nan_to_num(sem_first_half, nan=0)
    sem_second_half_clean = np.nan_to_num(sem_second_half, nan=0)
    
    plt.figure(figsize=(6, 4))
    data = [sem_first_half_clean, sem_second_half_clean]
    plt.boxplot(data, positions=[0, 1], widths=0.6, patch_artist=True, 
                boxprops=dict(facecolor='lightblue', color='blue'), 
                whiskerprops=dict(color='blue', linewidth=1), 
                capprops=dict(color='blue', linewidth=1), 
                flierprops=dict(marker='o', markerfacecolor='red', markersize=8, linestyle='none'))
    # Overlay the points for each run's SEM value
    for i in range(len(sem_first_half_clean)):
        # Plot SEM for the first half
        if sem_first_half_clean[i] > 0:  # Only plot if valid SEM value
            plt.scatter(0, sem_first_half_clean[i], color='blue', label='First Half' if i == 0 else "")
        
        # Plot SEM for the second half
        if sem_second_half_clean[i] > 0:  # Only plot if valid SEM value
            plt.scatter(1, sem_second_half_clean[i], color='red', label='Second Half' if i == 0 else "")

    plt.xlabel('Track Half')
    plt.ylabel('Standard Error of the Mean (SEM)')
    plt.title('Standard Error of the Mean for First and Second Halves of Test Runs')
    plt.xticks([0, 1], ['First Half', 'Second Half'])
    plt.legend()
    plt.grid(True, axis='y')
    plt.tight_layout()
    plt.show()

In [15]:
from scipy.stats import wilcoxon
def paired_test(values_first_half, values_second_half):
    """
    Performs a Wilcoxon signed-rank test on paired data.

    Parameters:
    -----------
    values_first_half : array-like, shape (n_samples, n_features)
        The first set of values, where each row represents an observation 
        and each column a feature.

    values_second_half : array-like, shape (n_samples, n_features)
        The second set of values to be compared with the first set.

    Returns:
    --------
    stat : float or None
        The computed Wilcoxon test statistic. Returns None if there 
        is insufficient data after filtering.

    p : float or None
        The p-value of the test, indicating the probability of observing 
        the given data under the null hypothesis. Returns None if there 
        is insufficient data after filtering.
    """
    mean_first_half = np.nanmean(values_first_half, axis=1)
    mean_second_half = np.nanmean(values_second_half, axis=1)
    # Remove NaN pairs
    valid_indices = ~np.isnan(mean_first_half) & ~np.isnan(mean_second_half)
    filtered_first_half = mean_first_half[valid_indices]
    filtered_second_half = mean_second_half[valid_indices]
    # Perform the Wilcoxon test
    if len(filtered_first_half) > 0:  # Ensure there's enough data for the test
        stat, p = wilcoxon(filtered_first_half, filtered_second_half)
        print(f"Wilcoxon test statistic: {stat}, p-value: {p}")
        return stat, p
    else:
        print("Not enough data after removing NaN values.")
        return None, None

In [13]:
def plot_means_test_runs_means(means_first_half, means_second_half):
    """
    Plots the means for the first and second halves of each test run on the x-axis,
    with the mean values on the y-axis, using two points for each test run: one for the first half
    and one for the second half.
    
    Parameters:
    - all_values_first_half: 2D array of values for the first half of each test run.
    - all_values_second_half: 2D array of values for the second half of each test run.
    """
    plt.figure(figsize=(6, 4))
    x_positions = [0, 1]  # Use 0 for 'First Half' and 1 for 'Second Half'
    for i in range(len(means_first_half)):
        plt.scatter(x_positions[0], means_first_half[i], color='blue', label='First Half' if i == 0 else "")  # Plot first half value
        plt.scatter(x_positions[1], means_second_half[i], color='red', label='Second Half' if i == 0 else "")  # Plot second half value

    for i in range(len(means_first_half)):  # Connect the points for the same test run with a line
        plt.plot([x_positions[0], x_positions[1]], [means_first_half[i], means_second_half[i]], color='black', linewidth=1)
        
    plt.xticks(x_positions, ['First Half', 'Second Half'])  # Set the x-tick labels to 'First Half' and 'Second Half'
    plt.xlabel('Track Half')
    plt.ylabel('Mean Value')
    plt.title('Mean Values for First and Second Halves of Test Runs')
    plt.legend()
    plt.grid(True, axis='y')
    plt.tight_layout()
    plt.show()


def plot_means_sem_first_second_half_means(means_first_half, means_second_half):
    """
    Plots the means and standard error of the mean (SEM) for the first and second halves
    of each test run, using box plots and points for each test run.
    
    Parameters:
    - means_first_half: 2D array of values for the first half of each test run.
    - means_second_half: 2D array of values for the second half of each test run.
    """
    
    # Calculate the SEM for each test run in the first and second halves
    sem_first_half = np.array([calculate_sem(values) for values in means_first_half])
    sem_second_half = np.array([calculate_sem(values) for values in means_second_half])

    # Clean the data: Replace NaN with 0 (or handle differently based on your needs)
    sem_first_half_clean = np.nan_to_num(sem_first_half, nan=0)
    sem_second_half_clean = np.nan_to_num(sem_second_half, nan=0)
    
    plt.figure(figsize=(6, 4))
    data = [sem_first_half_clean, sem_second_half_clean]
    plt.boxplot(data, positions=[0, 1], widths=0.6, patch_artist=True, 
                boxprops=dict(facecolor='lightblue', color='blue'), 
                whiskerprops=dict(color='blue', linewidth=1), 
                capprops=dict(color='blue', linewidth=1), 
                flierprops=dict(marker='o', markerfacecolor='red', markersize=8, linestyle='none'))
    # Overlay the points for each run's SEM value
    for i in range(len(sem_first_half_clean)):
        # Plot SEM for the first half
        if sem_first_half_clean[i] > 0:  # Only plot if valid SEM value
            plt.scatter(0, sem_first_half_clean[i], color='blue', label='First Half' if i == 0 else "")
        
        # Plot SEM for the second half
        if sem_second_half_clean[i] > 0:  # Only plot if valid SEM value
            plt.scatter(1, sem_second_half_clean[i], color='red', label='Second Half' if i == 0 else "")

    plt.xlabel('Track Half')
    plt.ylabel('Standard Error of the Mean (SEM)')
    plt.title('Standard Error of the Mean for First and Second Halves of Test Runs')
    plt.xticks([0, 1], ['First Half', 'Second Half'])
    plt.legend()
    plt.grid(True, axis='y')
    plt.tight_layout()
    plt.show()
    
def paired_test_mean (mean_first_half, mean_second_half):
    # valid_indices = ~np.isnan(mean_first_half) & ~np.isnan(mean_second_half)
    # filtered_first_half = mean_first_half[valid_indices]
    # filtered_second_half = mean_second_half[valid_indices]
    # Perform the Wilcoxon test
    if len(mean_first_half) > 0:  # Ensure there's enough data for the test
        stat, p = wilcoxon(mean_first_half, mean_second_half)
        print(f"Wilcoxon test statistic: {stat}, p-value: {p}")
    else:
        print("Not enough data after removing NaN values.")